In [1]:
from pathlib import Path

from openai import AsyncOpenAI
from rich_struct import print_obj

from mini_rag.ask import resolve_citations
from mini_rag.config import RetrieveConfig
from mini_rag.llm import OpenAIAnswerGenerator
from mini_rag.models import LlmAnswer
from mini_rag.prompts import build_context
from mini_rag.retrieve import retrieve_async

In [2]:
from mini_rag.models import Chunk


async def answer_question(question: str) -> tuple[LlmAnswer, dict[int, Chunk]]:
    config = RetrieveConfig(
        question=question,
        log_folder=Path("../logs"),
        index_dir=Path("../index/text-embedding-3-small"),
        chunks=Path("../index/chunks_600_90.jsonl"),
    )

    async with AsyncOpenAI() as client:
        search_results = await retrieve_async(config=config, client=client)

    question_context = build_context(search_results)

    async with AsyncOpenAI() as client:
        answer_provider = OpenAIAnswerGenerator(client, "gpt-5.4-mini")

        answer = await answer_provider.generate_async(question, question_context)

    return answer, question_context[1]

In [3]:
answer, ref_map = await answer_question("What is the purpose of IPCC Working Group III?")

17:28:42.034 INFO     Loading embeddings manifest from ..\index\text-embedding-3-small\embedding_manifest.json
17:28:42.054 INFO     Loading normalized embeddings from ..\index\text-embedding-3-small\embeddings.npy
17:28:42.071 INFO     Reading chunks from ..\index\chunks_600_90.jsonl
17:28:43.171 DEBUG    Searching for text What is the purpose of IPCC Working Group III?
17:28:43.172 INFO     Calling OpenAI embeddings (model=text-embedding-3-small) with 1 strings [Batch: 1]
17:28:43.780 DEBUG    OpenAI returned 1 embeddings (total_tokens: 10) [Batch: 1]
17:28:43.781 DEBUG    Semantic searching
17:28:43.795 INFO     Quering OpenAI's model 'gpt-5.4-mini'
17:28:48.306 INFO     OpenAI model 'gpt-5.4-mini' responded in 4.51 seconds (input_tokens: 3099, output_tokens: 45, total_tokens: 3144) [Response: resp_09792407dffa5275006afeb33bcad087d0a2b517c1e398d4d2]

In [4]:
print_obj(answer)

LlmAnswer(                                                                                                         
    answer='IPCC Working Group III assesses mitigation response options, meaning options for mitigating climate    
change [1].',                                                                                                      
    cited_refs=[                                                                                                   
        1                                                                                                          
    ],                                                                                                             
    insufficient_evidence=False                                                                                    
)                                                                                                                  

In [5]:
answer2, ref_map2 = await answer_question("How much did Apple earn in fiscal year 2025?")

17:28:48.327 INFO     Loading embeddings manifest from ..\index\text-embedding-3-small\embedding_manifest.json
17:28:48.344 INFO     Loading normalized embeddings from ..\index\text-embedding-3-small\embeddings.npy
17:28:48.358 INFO     Reading chunks from ..\index\chunks_600_90.jsonl
17:28:48.991 DEBUG    Searching for text How much did Apple earn in fiscal year 2025?
17:28:48.992 INFO     Calling OpenAI embeddings (model=text-embedding-3-small) with 1 strings [Batch: 1]
17:28:49.471 DEBUG    OpenAI returned 1 embeddings (total_tokens: 12) [Batch: 1]
17:28:49.473 DEBUG    Semantic searching
17:28:49.484 INFO     Quering OpenAI's model 'gpt-5.4-mini'
17:28:53.742 INFO     OpenAI model 'gpt-5.4-mini' responded in 4.26 seconds (input_tokens: 3145, output_tokens: 44, total_tokens: 3189) [Response: resp_0a4ed3a9e2105136006afeb3413af887d082eb7751a4175ab6]

In [6]:
print_obj(answer2)

LlmAnswer(                                                                                                         
    answer="I'm sorry but based on provided document fragments I can't answer the question",                       
    cited_refs=[],                                                                                                 
    insufficient_evidence=True                                                                                     
)                                                                                                                  

## Resolving

In [7]:
resolved_citations = resolve_citations(answer, ref_map)

In [8]:
print_obj(resolved_citations)

ResolvedCitations(                                                                                                 
    resolved_citations=[                                                                                           
        Citation(                                                                                                  
            ref=1,                                                                                                 
            chunk_id='WGI_CH01_p009_c001',                                                                         
            document_id='WGI_CH01',                                                                                
            page=9,                                                                                                
            page_end=9                                                                                             
        )                                                                                                          
    ],                                                                                                             
    non_empty_citations_error=False,                                                                               
    mismatched_citations=[],                                                                                       
    extra_references=[]                                                                                            
)                                                                                                                  